# Indirect prompt injection in LLM CV screening: analysis
Replication of Milani, Franzoni & Florindi (2026), *Neural Computing & Applications* 38:530, plus three extra defenses.

Set `RESULTS` to the JSONL written by `runner.py`. All logic lives in `analysis.py` / `metrics.py`, so the notebook and the CLI (`python analysis.py`) give identical numbers.

In [ ]:
from pathlib import Path
import pandas as pd
import analysis as A
from metrics import score_file, false_alarm_rate, validity
from llm_clients import load_config

cfg = load_config('config.yaml')
RESULTS = Path(cfg['experiment']['results'])   # e.g. Path('results/raw.jsonl')
N_BOOT = 2000
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

In [ ]:
cand, trials = score_file(RESULTS, cfg['experiment']['cv_dir'], cfg['metrics']['asr_threshold'],
                          cfg['metrics']['baseline_defense'])
if set(trials['model']) <= {'mock'}:
    print('WARNING: mock-simulator data - not experimental results')
print(f"{len(trials):,} attacked trials · models: {sorted(trials['model'].unique())} · "
      f"defenses: {sorted(trials['defense'].unique())}")
trials.head()

## Table 4: per model
ASR without vs with each defense (95% CI, bootstrap resampling whole CVs), ME, DAR.

In [ ]:
A.table4(trials, N_BOOT)

## Table 5: by seniority (document length)
Paper: defended ASR 15% (Junior) vs 33% (Senior), which the authors attribute to the 'lost in the middle' effect.

In [ ]:
A.table5(trials, N_BOOT)

## Table 6: by achievement density
Paper: high-AD CVs show less bias (ΔSI 1.2 vs 2.6) but are harder to fully mitigate (ME 71% vs 86%).

In [ ]:
A.table6(trials, N_BOOT)

## Table 7: by injection type
`aware_bypassed` is the share of trials where the model raised an alert *and* the attack still succeeded (paper: p.Pos-Med detected 85.5% of the time, yet 32.2% ASR).

In [ ]:
A.table7(trials, N_BOOT)

## False alarms on clean CVs
How often each defense raises an alert when there is no injection.

In [ ]:
false_alarm_rate(cand)

## Regression
Logistic regression of success on model, seniority, AD class, attack type and defense, with cluster-robust SEs by CV, plus a Bayesian mixed-effects logit with a random intercept per CV. A defense with 0% ASR causes perfect separation; read its raw rate, not its odds ratio.

In [ ]:
reg = A.regression(trials)
print(reg['formula'])
reg.get('odds_ratios', reg.get('note') or reg.get('logit_error'))

In [ ]:
reg['mixed'].summary() if 'mixed' in reg else reg.get('mixed_error')

## Charts

In [ ]:
A.fig_asr_bars(trials, n_boot=1000);

In [ ]:
for d in sorted(trials['defense'].unique()):
    A.fig_heatmap(trials, d)

In [ ]:
A.fig_me_vs_dar(trials) if (trials['defense'] != 'none').any() else None;

## Lexicon SI vs LLM-as-judge (validity check)
Run `python metrics.py --judge-model <key>` first; this cell only reads the cached judge scores if they exist.

In [ ]:
from metrics import judge_scores, load_results
from llm_clients import load_clients
JUDGE = None   # e.g. 'llama3'; scores are cached in results/judge_cache.jsonl
if JUDGE:
    c = judge_scores(cand, load_results(RESULTS), load_clients('config.yaml', [JUDGE])[JUDGE],
                     RESULTS.parent / 'judge_cache.jsonl')
    print(validity(c))

## Extension: which defense works best?
Compares OPE with spotlighting/datamarking, the two-pass JSON sanitizer and the hidden-text and imperative-sentence pre-filter, for p.Pos-Med and for Senior CVs.

In [ ]:
from IPython.display import Markdown
report = A.defense_report(trials, cand, N_BOOT, source=str(RESULTS))
Path('results/defense_report.md').write_text(report, encoding='utf-8')
Markdown(report)